# ForecastLab — engine walkthrough

**Developed by Wallison Sampaio** — Commercial Intelligence | Supply Chain | Data & AI Analytics Specialist · [LinkedIn](https://www.linkedin.com/in/wallisonsampaio)

This notebook uses the same modules as the FastAPI backend to show the pipeline on the synthetic dataset:
generate data → profile a series → route candidate models → rolling-origin backtest → champion / challenger → P10/P50/P90.

Run it from the `backend/` virtual environment (`pip install jupyter`).

In [ ]:
import sys
sys.path.insert(0, '../backend')

from synthetic_data_generator import generate_dataset
from app.services.datasets import _canonical_from_generator
from app.forecasting.panel import build_panel
from app.forecasting.profiling import profile_series
from app.forecasting.router import candidate_models
from app.forecasting.worker import RunParams, process_series

ds = generate_dataset(random_seed=42)
print(ds.metadata['n_rows'], 'rows ·', ds.metadata['n_series'], 'series ·', ds.metadata['start'], '→', ds.metadata['end'])
ds.sales.head()

In [ ]:
sales = _canonical_from_generator(ds.sales)
panel = {s.series_id: s for s in build_panel(sales, horizon=12)}
s = panel['Atlas Hipermercados|SKU-30102']  # sunscreen: strong summer seasonality
p = profile_series(s.y, s.X['promotion'].to_numpy(), s.X['distribution'].to_numpy())
print(p['series_type'], '—', p['classification_reason'])
print('candidates:', candidate_models(p['series_type']))

In [ ]:
import pandas as pd

res = process_series(s, RunParams())
board = pd.DataFrame(res['registry'])[['rank', 'model', 'wmape', 'bias', 'rmse', 'mase', 'status']]
print('Champion:', res['champion'], '· Challenger:', res['challenger'], '· Confidence:', res['confidence'])
board.sort_values('rank')

In [ ]:
fc = pd.DataFrame(res['forecasts'])
fc[fc['role'] == 'champion'][['date', 'p10', 'p50', 'p90']].round(0)